# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [1]:
import sys
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [2]:
# ------------------------------- setting start ------------------------------ #
# RULE: no plot in this notebook gets a suptitle/figure title. Captions live in the
# LaTeX paper, not the PNG. Per-panel titles inside a multi-scene grid (plot_grid's
# ax.set_title(scene)) are the only exception -- they label panels, not the figure.
# Quick throwaway/debug plots outside this notebook can have titles; paper figures can't.
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'serif', 'serif': ['Times New Roman', 'Times'], 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
plt.rcParams['text.latex.preamble'] = r'\usepackage{mathptmx}'
# -------------------------------- setting end ------------------------------- #

In [3]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-13: exp1 is a slice of the comprehensive Exp4 grid sweep (grid8, progressive
# packing, vd_lod scheme, weight_mode as the swept dimension) -- not a separate run. Using
# the same fixed-ml/lpips-enabled CSV as Exp4 instead of the old standalone exp1 CSV, which
# predates the ml label fix and has no LPIPS.
SUMMARY_CSV = "output/0704/quality_sweep/summary_all_ml_fixed.csv"
OUT_DIR     = "plotting/paper/exp1_weights/"
GROUP_BY    = "weight_mode"
EXCLUDE_KEYS = ["random"]
SLICE_FILTERS = {"grid_shape": "[8, 8, 8]", "packing_mode": "progressive", "scheme": "vd_lod"}

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

color_palette = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"]

# canon (2026-07-03): sum-w_mode exp2 rerun showed vd_lod_w (with distance term) craters on
# real scenes -- v_lod_w (W_k only, no distance) is now THE deployed heuristic ("ours").
# vd_lod_w kept in KEY_CONFIG for historical/retired reference, excluded from exp2 plots.
KEY_CONFIG = {
    "screen_area":      {"label": "Screen Area",                   "marker": "o", "color": color_palette[4]},
    "volume":           {"label": "Volume (view-indep.)",          "marker": "^", "color": color_palette[2]},
    "volume_over_d2":   {"label": r"Volume/$d^2$ (view-dep.)",        "marker": "x", "color": color_palette[3]},
    # "random":           {"label": "Random (control)",              "marker": "x", "color": color_palette[7]},
    "vd_lod":           {"label": "Heuristic (baseline)",             "marker": "s", "color": color_palette[0]},
    "vd_lod_w":         {"label": "Heuristic (ours, retired)",             "marker": "^", "color": color_palette[1]},
    "v_lod_w":          {"label": "Heuristic (ours)",         "marker": "v", "color": color_palette[5]},
    "ml":               {"label": "Learned (ours)",                     "marker": "P", "color": color_palette[2]},
    "oracle_loo":       {"label": "Proxy Oracle",       "marker": "*", "color": color_palette[3]},
    "prog_cull":        {"label": "GS (culled)",    "marker": "o", "color": color_palette[0]},
    "prog_no_cull":     {"label": "GS (not culled)",      "marker": "^", "color": color_palette[2]},
    "tile_strict":      {"label": "Tiles",                   "marker": "s", "color": color_palette[6]},
    "tiled_ml":         {"label": "Tiles Learned",        "marker": "P", "color": color_palette[1]},
    "tiled_oracle":     {"label": "Tiles Oracle",      "marker": "*", "color": color_palette[3]},
}

KEY_ORDER = {
    "weight_mode": ["screen_area", "volume", "volume_over_d2", "random"],
    "scheme":      ["vd_lod", "v_lod_w", "ml", "oracle_loo"],
    "condition":   ["prog_cull", "prog_no_cull", "tiled_ml", "tiled_oracle"],
}

DPI = 300


In [4]:
import os, sys

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
_candidates = [_cwd] + list(_cwd.parents) + [Path(p) for p in sys.path]
for _candidate in _candidates:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

cwd: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


In [5]:
# shared DATA pipeline: single source of truth with experiments/plot_metrics.py
# (rendering is paper-specific and lives in this notebook, like plot_metric below)
sys.path.insert(0, str(Path.cwd()))
from experiments.plot_metrics import (
    _apply_budget_labels, _aggregate, _resolve_order_and_labels,
    _data_ylim, _bk_sort, PSNR_SATURATION_DB,
)

In [6]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
for _col, _val in SLICE_FILTERS.items():
    df = df[df[_col] == _val]
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

# convert to list-of-dicts (plot_metrics format) and attach per-scene budget labels
rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 36000 rows | groups: ['screen_area', 'volume', 'volume_over_d2'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [7]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

# paper ordering: prefer KEY_ORDER override, fall back to plot_metrics order
preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

# labels: KEY_CONFIG overrides plot_metrics defaults
labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

# per-scene aggregates for grid plot
from collections import defaultdict as _dd
_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
SCENE_ORDER = ["bicycle", "garden", "stump", "chair", "drums", "ficus", "hotdog", "materials", "mic", "ship"]
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

sample_key = order[0]
sample_bk  = sorted(agg[sample_key], key=_bk_sort)[0]
print(f"agg sample  {GROUP_BY}={sample_key!r}  budget={sample_bk}:")
print(agg[sample_key][sample_bk])
print(f"scenes: {list(scenes_agg.keys())}")

agg sample  weight_mode='screen_area'  budget=10%:
{'psnr_mean': 26.4497313934962, 'psnr_ci95': 2.386178168493047, 'ssim_mean': 0.871449022307992, 'ssim_ci95': 0.02316555374384588, 'lpips_mean': 0.16373598393900513, 'lpips_ci95': 0.030692025026235067, 'ngs_mean': 187667.6, 'ngs_ci95': 160051.17141777815, 'n': 10, 'n_cameras': 1500}
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [8]:
def plot_metric(agg, order, metric, ylabel, out_stem, out_dir, key_config=None):
    key_config = KEY_CONFIG if key_config is None else key_config
    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)
    all_means = []

    for i, key in enumerate(order):
        if key not in agg:
            continue
        cfg    = key_config.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])
        ls     = cfg.get("ls", "-")

        bks = sorted(agg[key], key=_bk_sort)
        xs  = [_bk_sort(bk) for bk in bks]
        ys  = [agg[key][bk][f"{metric}_mean"] for bk in bks]
        es  = [agg[key][bk][f"{metric}_ci95"]  for bk in bks]
        all_means.extend(ys)

        ax.errorbar(xs, ys, yerr=es,
                    marker=marker, color=color, linestyle=ls, linewidth=2.5, markersize=8,
                    capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
                    label=label, zorder=2)

    ax.set_xlabel(r"Budget (\% of scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
    ax.set_ylim(*_data_ylim(all_means, metric))
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(ax.get_xlim()[1], PSNR_SATURATION_DB, r" saturation ($\geq$60 dB)",
                fontsize=13, color="gray", va="bottom", ha="right")
    ax.legend(loc="best", framealpha=0.9, fontsize=18)
    ax.tick_params(labelsize=18)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.set_constrained_layout(True)
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [9]:
def plot_bar(agg, order, metric, ylabel, out_path):
    """Cross-scene grouped bars, paper style (tab10 KEY_CONFIG colors, no title/grid)."""
    bks    = sorted({b for k in agg for b in agg[k]}, key=_bk_sort)
    groups = [k for k in order if k in agg]
    n_b, n_g = len(bks), len(groups)
    width  = 0.8 / max(n_g, 1)
    x      = np.arange(n_b)

    fig, ax = plt.subplots(figsize=(max(8.0, n_b * 1.3), 4.8))
    all_means = []
    for i, key in enumerate(groups):
        cfg = KEY_CONFIG.get(key, {})
        y   = [agg[key].get(b, {}).get(f"{metric}_mean", 0.0) for b in bks]
        err = [agg[key].get(b, {}).get(f"{metric}_ci95",  0.0) for b in bks]
        all_means.extend(v for b, v in zip(bks, y) if b in agg[key])
        offset = (i - n_g / 2 + 0.5) * width
        ax.bar(x + offset, y, width * 0.92, yerr=err, capsize=err_capsize,
               color=cfg.get("color", color_palette[i % len(color_palette)]),
               label=cfg.get("label", key),
               error_kw={"elinewidth": err_lw, "capthick": err_capthick})

    ax.set_ylim(*_data_ylim(all_means, metric))
    ax.set_xticks(x)
    ax.set_xticklabels([b.replace("%", r"\%") for b in bks])
    ax.set_xlabel(r"Budget (\% of scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=16, framealpha=0.9)
    ax.set_axisbelow(True)
    if metric == "psnr":
        ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.5, zorder=0)
        ax.text(n_b - 0.5, PSNR_SATURATION_DB, r"saturation ($\geq$60 dB) ",
                fontsize=13, color="gray", va="bottom", ha="right")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [10]:
def plot_grid(scenes_agg, order, metric, ylabel, out_path, ncols=5, key_config=None):
    """Per-scene subplot grid, paper style (tab10 colors, shared top legend, no clutter).
    `scenes_agg` keys are panel titles; `key_config` defaults to KEY_CONFIG (scheme
    lines) -- pass GRID_CONFIG when lines represent grid size instead of scheme."""
    key_config = KEY_CONFIG if key_config is None else key_config
    scenes = list(scenes_agg.keys())
    n      = len(scenes)
    ncols  = min(ncols, n)
    nrows  = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.4, nrows * 3.0),
                             squeeze=False, sharey=True, constrained_layout=True)
    axes_flat = axes.flatten()
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    ylim = _data_ylim(
        [cell[f"{metric}_mean"]
         for a in scenes_agg.values()
         for budgets in a.values()
         for cell in budgets.values()],
        metric,
    )

    for idx, scene in enumerate(scenes):
        ax = axes_flat[idx]
        a  = scenes_agg[scene]
        bks = next((sorted(a[k], key=_bk_sort) for k in order if k in a), None)
        if bks is None:
            continue
        xs = [_bk_sort(b) for b in bks]
        for key in (k for k in order if k in a):
            cfg = key_config.get(key, {})
            y   = [a[key].get(b, {}).get(f"{metric}_mean", float("nan")) for b in bks]
            err = [a[key].get(b, {}).get(f"{metric}_ci95", 0.0) for b in bks]
            ax.errorbar(xs, y, yerr=err,
                        marker=cfg.get("marker", "o"),
                        color=cfg.get("color", color_palette[0]),
                        linestyle=cfg.get("ls", "-"),
                        linewidth=2, markersize=5, capsize=err_capsize,
                        label=cfg.get("label", key))
        ax.set_title(scene, fontsize=14)
        ax.set_ylim(*ylim)
        if metric == "psnr":
            ax.axhline(PSNR_SATURATION_DB, color="gray", linestyle=":", linewidth=1.0, zorder=0)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
        ax.tick_params(labelsize=12)
        if idx % ncols == 0:
            ax.set_ylabel(ylabel, fontsize=14)

    # one shared legend on top, collected across panels
    seen = {}
    for ax in axes_flat[:n]:
        for h, l in zip(*ax.get_legend_handles_labels()):
            seen.setdefault(l, h)
    fig.legend(list(seen.values()), list(seen.keys()),
               loc="upper center", ncol=len(seen), fontsize=14,
               framealpha=0.9, bbox_to_anchor=(0.5, 1.12))
    fig.supxlabel(r"Budget (\% of scene)", fontsize=16)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [11]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (lower is better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (lower is better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp1_weights/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp1_weights/_grid/lpips_vs_budget.png
Done.


## Exp 2 — Scheme Ranking

Slice of the comprehensive Exp4 grid sweep (grid8, tile_partial packing, screen_area
weight_mode, scheme as the swept dimension) -- not a separate run. Source:
`output/0704/quality_sweep/summary_all_ml_fixed.csv` (fixed-ml/lpips-enabled, 2026-07-13).
Schemes: `vd_lod` (baseline), `v_lod_w` (heuristic, ours), `ml` (learned, ours),
`oracle_loo` (upper bound). 10 scenes x 150 eval cams.


In [12]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-13: exp2 is a slice of the comprehensive Exp4 grid sweep (grid8, tile_partial
# packing, screen_area weight_mode, scheme as the swept dimension) -- not a separate run.
# Using the same fixed-ml/lpips-enabled CSV as Exp4/Exp1 instead of the old standalone
# exp2 CSV, which predates the ml label fix and has no LPIPS.
SUMMARY_CSV = "output/0704/quality_sweep/summary_all_ml_fixed.csv"
OUT_DIR     = "plotting/paper/exp2/"
GROUP_BY    = "scheme"
EXCLUDE_KEYS = ["vd_lod_w"]  # retired 2026-07-03 -- v_lod_w is canonical "ours" heuristic
SLICE_FILTERS = {"grid_shape": "[8, 8, 8]", "packing_mode": "tile_partial", "weight_mode": "screen_area"}

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
for _col, _val in SLICE_FILTERS.items():
    df = df[df[_col] == _val]
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")


Loaded 48000 rows | groups: ['ml', 'oracle_loo', 'v_lod_w', 'vd_lod'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']


In [13]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")

order: ['vd_lod', 'v_lod_w', 'ml', 'oracle_loo']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [14]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (lower is better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (lower is better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp2/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp2/_grid/lpips_vs_budget.png
Done.


## Exp 3 — GS ordering vs Tiled selection (established 4-way comparison)

Source: `output/0704/quality_sweep/summary_all_ml_fixed.csv`, same comprehensive sweep as
Exp1/Exp2/Exp4. Established pattern (originally `output/quickplots/0606/exp123_merged/`,
deleted 2026-06-28 per PLAN.md's wrap-up log; `condition` KEY_CONFIG/KEY_ORDER entries and
`experiments/plot_metrics.py`'s `_SCHEME_LABELS` `prog_no_cull`/`prog_culled`/`oracle_tp`/`ml_tp`
kept the naming alive) -- 4 fixed conditions, each a specific (grid, packing_mode, scheme)
point, not a raw column slice:

- `prog_cull`   = grid8,  progressive,   vd_lod      (GS ordering, culled)
- `prog_no_cull`= grid1,  progressive,   vd_lod      (GS ordering, not culled)
- `tiled_oracle`= grid8,  tile_partial,  oracle_loo  (tiled, oracle upper bound)
- `tiled_ml`    = grid8,  tile_partial,  ml          (tiled, learned)


In [15]:
# ── I/O ──────────────────────────────────────────────────────────────────────
SUMMARY_CSV = "output/0704/quality_sweep/summary_all_ml_fixed.csv"
OUT_DIR     = "plotting/paper/exp3_gs_vs_tiled/"
GROUP_BY    = "condition"
EXCLUDE_KEYS = []

BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

# 4 fixed (grid, packing_mode, scheme) points, flattened into one "condition" column --
# matches the established prog_cull/prog_no_cull/tiled_oracle/tiled_ml naming (KEY_CONFIG
# above, experiments/plot_metrics.py's _SCHEME_LABELS).
EXP3_CONDITIONS = {
    "prog_cull":    ("[8, 8, 8]", "progressive",  "vd_lod"),
    "prog_no_cull": ("[1, 1, 1]", "progressive",  "vd_lod"),
    "tiled_oracle": ("[8, 8, 8]", "tile_partial", "oracle_loo"),
    "tiled_ml":     ("[8, 8, 8]", "tile_partial", "ml"),
}

summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df_full = pd.read_csv(summary_csv)
df_full = df_full[df_full["weight_mode"] == "screen_area"]

parts = []
for cond, (grid, packing, scheme) in EXP3_CONDITIONS.items():
    sub = df_full[(df_full["grid_shape"] == grid) & (df_full["packing_mode"] == packing)
                  & (df_full["scheme"] == scheme)].copy()
    sub["condition"] = cond
    parts.append(sub)
df = pd.concat(parts, ignore_index=True)

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

rows = df.to_dict("records")
rows = _apply_budget_labels(rows, BUDGET_PCTS)

print(f"Loaded {len(rows)} rows | groups: {sorted(set(r[GROUP_BY] for r in rows))} | budgets: {sorted(set(r['_budget_key'] for r in rows), key=_bk_sort)}")
for cond, (grid, packing, scheme) in EXP3_CONDITIONS.items():
    n = sum(1 for r in rows if r["condition"] == cond)
    print(f"  {cond:14s} = grid{grid} {packing:14s} {scheme:10s} -> {n} rows")


Loaded 48000 rows | groups: ['prog_cull', 'prog_no_cull', 'tiled_ml', 'tiled_oracle'] | budgets: ['10%', '25%', '40%', '55%', '70%', '85%', '99%', '100%']
  prog_cull      = grid[8, 8, 8] progressive    vd_lod     -> 12000 rows
  prog_no_cull   = grid[1, 1, 1] progressive    vd_lod     -> 12000 rows
  tiled_oracle   = grid[8, 8, 8] tile_partial   oracle_loo -> 12000 rows
  tiled_ml       = grid[8, 8, 8] tile_partial   ml         -> 12000 rows


In [16]:
agg    = _aggregate(rows, GROUP_BY)
pm_order, pm_labels = _resolve_order_and_labels(agg, GROUP_BY)

preferred = KEY_ORDER.get(GROUP_BY, pm_order)
order = [k for k in preferred if k in agg] + [k for k in pm_order if k not in preferred and k in agg]

labels = {k: KEY_CONFIG[k]["label"] if k in KEY_CONFIG else pm_labels.get(k, k) for k in order}

_by_scene = _dd(list)
for r in rows:
    _by_scene[r["scene"]].append(r)
scenes_agg = {s: _aggregate(_by_scene[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene}

print(f"order: {order}")
print(f"scenes: {list(scenes_agg.keys())}")


order: ['prog_cull', 'prog_no_cull', 'tiled_ml', 'tiled_oracle']
scenes: ['bicycle', 'garden', 'stump', 'chair', 'drums', 'ficus', 'hotdog', 'materials', 'mic', 'ship']


In [17]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# 1. aggregate line plot (paper main figure)
plot_metric(agg, order, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, order, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
plot_metric(agg, order, "lpips", r"LPIPS (lower is better)", "lpips_vs_budget", out_dir)

# 2. per-scene grid
plot_grid(scenes_agg, order, "psnr", r"Quality in PSNR (dB)", grid_dir / "psnr_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "ssim", r"Quality in SSIM", grid_dir / "ssim_vs_budget.png", ncols=5)
plot_grid(scenes_agg, order, "lpips", r"LPIPS (lower is better)", grid_dir / "lpips_vs_budget.png", ncols=5)
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/ssim_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp3_gs_vs_tiled/_grid/psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/_grid/ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp3_gs_vs_tiled/_grid/lpips_vs_budget.png
Done.


## Wall-Time / Selection Latency (Exp5)

Per-frame selection cost by condition. Two-stage aggregation (scene -> group), same
principle as the PSNR/SSIM `_aggregate` above: scenes are the independent statistical
unit, not cameras, so CI is computed over `n_scenes`, not pooled cameras.

Source: `time_selection.py` full sweep, `../VCUTS_backup/output/0702/selection_timing/{scene}/{method}/summary.csv`
(150 cams/method; archived 2026-07-13, see PLAN.md's backup-move note). Scene roster is a
static dict (`TIMING_SCENE_GROUP`, not discovered from the sweep directory) — extending to
more scenes means adding a `summary.csv` under an already-listed scene name; scenes without
one yet are skipped with a printed warning.


In [18]:
import matplotlib.patches as mpatches

# ── config ───────────────────────────────────────────────────────────────────
# suffix -> (sweep dir, output dir). "" (grid8, canonical) writes to the paper dir with
# unsuffixed filenames. grid2/grid4 (2026-07-03 retry, PLAN.md item vii addendum) are an
# exploratory retry, not paper-canonical -- they go to output/quickplots/, not the paper dir.
TIMING_SWEEPS = {
    "":       (Path("../VCUTS_backup/output/0702/selection_timing"),        Path("plotting/paper/timings_selection")),
    "_grid2": (Path("../VCUTS_backup/output/0703/selection_timing/grid2"),  Path("output/quickplots/0703_timing_grid_retry")),
    "_grid4": (Path("../VCUTS_backup/output/0703/selection_timing/grid4"),  Path("output/quickplots/0703_timing_grid_retry")),
}

# full canonical roster (same 10 scenes as SCENE_ORDER above). Only chair/bicycle
# have a timing sweep run so far -- the rest are skipped at load time until they do.
TIMING_SCENE_GROUP = {
    "chair": "synth", "drums": "synth", "ficus": "synth", "hotdog": "synth",
    "materials": "synth", "mic": "synth", "ship": "synth",
    "bicycle": "real", "garden": "real", "stump": "real",
}
TIMING_GROUP_LABELS = {"synth": "Synthetic", "real": "Real"}

# condition -> method dir name (same for every scene, all use canonical "ml_lgbm" now).
# grid2/grid4 lack a tiled_ml sweep (output/ml_models/8/{scene}/AC was trained on grid8
# tiles, not valid at other grid sizes) -- tiled_ml simply won't appear in those bars.
TIMING_COND_METHOD = {
    "prog_screen_area": "progressive_screen_area",
    "prog_vol_over_d2": "progressive_vol_d2",
    "tiled_vd_lod":     "vd_lod",
    "tiled_heuristic":  "heuristic",
    "tiled_ml":         "ml_lgbm",
    "tiled_oracle":     "oracle_online",
}
# ml_rf excluded: ml_lgbm is faster on both scenes measured so far and is the
# canonical ML scheme since 2026-07-02 (PLAN.md).

TIMING_COND_LABELS = {
    "prog_screen_area": "Prog.\nScreen Area",
    "prog_vol_over_d2": r"Prog.$\ \mathrm{Vol}/d^2$",
    "tiled_vd_lod":     "Tiled\nBaseline",
    "tiled_heuristic":  "Tiled\nHeuristic",
    "tiled_ml":         "Tiled\nML",
    "tiled_oracle":     "Tiled\nOracle",
}
TIMING_COND_ORDER = list(TIMING_COND_METHOD.keys())

# "utility" = tile_weights_s (aggregate per-GS weight -> per-tile W_k) + utility_s
# (evaluate v/d*W_k) -- per time_selection.py these run back-to-back as one
# "produce a per-tile score" step (heuristic: gaussian_weights -> tile_weights ->
# utility). GS Weights stays separate: it's the upstream per-GS signal computation,
# same role as ML Features plays for the ML scheme.
# "render" (oracle's actual LOO renders) is a different operation that happens to
# share the same pipeline slot as "utility" -- kept as its own stage, not merged in.
# colors: reuse color_palette (defined above) by index, not an invented palette.
TIMING_STAGE_DEFS = [
    ("visibility", "Visibility",      color_palette[0]),
    ("gw",         "GS Weights",      color_palette[1]),
    ("ml_feat",    "ML Features",     color_palette[2]),
    ("utility",    "Tile Utility", color_palette[3]),
    ("render",     "LOO Render",      color_palette[4]),
    ("greedy_sel", "Greedy Select",   color_palette[5]),
]

In [19]:
# ── load per-camera timing summaries, one row per (scene, condition), per sweep ────
timing_dfs = {}
for _suffix, (_sweep_dir, _out_dir) in TIMING_SWEEPS.items():
    _timing_rows, _timing_missing = [], []
    for scene, group in TIMING_SCENE_GROUP.items():
        for cond, method in TIMING_COND_METHOD.items():
            csv = _sweep_dir / scene / method / "summary.csv"
            if not csv.exists():
                _timing_missing.append(f"{scene}/{cond}")
                continue
            mdf = pd.read_csv(csv).fillna(0.0)
            _timing_rows.append({
                "scene": scene, "group": group, "condition": cond, "n_cameras": len(mdf),
                "total_ms":   mdf["total_s"].mean() * 1000,
                "visibility": mdf["visibility_s"].mean() * 1000,
                "gw":         mdf["gaussian_weights_s"].mean() * 1000,
                "ml_feat":    (mdf["ml_group_a_s"] + mdf["ml_predict_s"]).mean() * 1000,
                "utility":    (mdf["tile_weights_s"] + mdf["utility_s"]).mean() * 1000,
                "render":     (mdf["full_render_s"] + mdf["tile_renders_s"]).mean() * 1000,
                "greedy_sel": mdf["greedy_s"].mean() * 1000,
            })
    timing_dfs[_suffix] = pd.DataFrame(_timing_rows)
    print(f"[sweep{_suffix or ':grid8'}] loaded {len(_timing_rows)} rows | scenes present: {sorted(timing_dfs[_suffix]['scene'].unique())}")
    if _timing_missing:
        print(f"[sweep{_suffix or ':grid8'}] not swept, skipped: {_timing_missing}")

timing_df = timing_dfs[""]  # kept for backward compat with any ad-hoc inspection below
timing_df.head()

[sweep:grid8] loaded 60 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']


[sweep_grid2] loaded 50 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']
[sweep_grid2] not swept, skipped: ['chair/tiled_ml', 'drums/tiled_ml', 'ficus/tiled_ml', 'hotdog/tiled_ml', 'materials/tiled_ml', 'mic/tiled_ml', 'ship/tiled_ml', 'bicycle/tiled_ml', 'garden/tiled_ml', 'stump/tiled_ml']
[sweep_grid4] loaded 50 rows | scenes present: ['bicycle', 'chair', 'drums', 'ficus', 'garden', 'hotdog', 'materials', 'mic', 'ship', 'stump']
[sweep_grid4] not swept, skipped: ['chair/tiled_ml', 'drums/tiled_ml', 'ficus/tiled_ml', 'hotdog/tiled_ml', 'materials/tiled_ml', 'mic/tiled_ml', 'ship/tiled_ml', 'bicycle/tiled_ml', 'garden/tiled_ml', 'stump/tiled_ml']


,scene,group,condition,n_cameras,total_ms,visibility,gw,ml_feat,utility,render,greedy_sel
0,chair,synth,prog_screen_area,150,7.917177,0.551486,4.126077,0.000000,0.000000,0.0,3.239613
1,chair,synth,prog_vol_over_d2,150,2.845473,0.548037,1.624857,0.000000,0.000000,0.0,0.672579
2,chair,synth,tiled_vd_lod,150,1.742173,0.548516,0.000000,0.000000,0.139095,0.0,1.054562
3,chair,synth,tiled_heuristic,150,8.652659,0.557242,4.144361,0.000000,2.993050,0.0,0.958005
4,chair,synth,tiled_ml,150,5.856278,0.568127,0.000000,4.266225,0.000000,0.0,1.021927


In [20]:
# ── aggregate: group mean per condition (+ CI once >1 scene/group exists), per sweep ──
def _ci95(vals):
    n = len(vals)
    return 1.96 * float(np.std(vals, ddof=1)) / np.sqrt(n) if n > 1 else 0.0


timing_bar_aggs = {}
for _suffix, _df in timing_dfs.items():
    _agg = {"synth": {}, "real": {}}
    for group in _agg:
        gdf = _df[_df["group"] == group]
        for cond in TIMING_COND_ORDER:
            vals = gdf.loc[gdf["condition"] == cond, "total_ms"].to_numpy()
            if len(vals) == 0:
                continue
            _agg[group][cond] = {"mean": float(np.mean(vals)), "ci95": _ci95(vals)}
    timing_bar_aggs[_suffix] = _agg

timing_bar_agg = timing_bar_aggs[""]  # kept for backward compat with any ad-hoc inspection below


def timing_stage_group_means(group, df=None):
    """Per-condition stage-mean ms, averaged across scenes in `group`. `df` defaults to
    the grid8 sweep for backward compat; pass timing_dfs[suffix] for other sweeps."""
    gdf = (df if df is not None else timing_df)
    gdf = gdf[gdf["group"] == group]
    return {
        cond: {sk: float(gdf.loc[gdf["condition"] == cond, sk].mean()) for sk, _, _ in TIMING_STAGE_DEFS}
        for cond in TIMING_COND_ORDER if cond in gdf["condition"].unique()
    }


print(timing_bar_agg)

{'synth': {'prog_screen_area': {'mean': 8.164160432560053, 'ci95': np.float64(1.1201709306576337)}, 'prog_vol_over_d2': {'mean': 2.898773891585165, 'ci95': np.float64(0.36595119368679463)}, 'tiled_vd_lod': {'mean': 1.8250795028038884, 'ci95': np.float64(0.13692240438504344)}, 'tiled_heuristic': {'mean': 8.887865651576243, 'ci95': np.float64(1.1641067935836467)}, 'tiled_ml': {'mean': 6.626102585522853, 'ci95': np.float64(1.497442637077164)}, 'tiled_oracle': {'mean': 2706.3852757375157, 'ci95': np.float64(1323.5703445036165)}}, 'real': {'prog_screen_area': {'mean': 232.13732183807429, 'ci95': np.float64(36.27802804466195)}, 'prog_vol_over_d2': {'mean': 95.99631837258733, 'ci95': np.float64(16.17134312173665)}, 'tiled_vd_lod': {'mean': 62.55741968750949, 'ci95': np.float64(6.122159506621896)}, 'tiled_heuristic': {'mean': 273.97820834898283, 'ci95': np.float64(37.13865690440117)}, 'tiled_ml': {'mean': 64.06058132648464, 'ci95': np.float64(9.218367566123353)}, 'tiled_oracle': {'mean': 438.8

In [21]:
def plot_timing_bar(bar_agg_group, cond_order, cond_labels, color, label, out_path, ylim):
    """Single-group bar, log y-axis (latency spans ~3 orders of magnitude: oracle vs the
    rest, in both synth and real). `ylim` is shared across synth/real calls so the two
    figures are visually comparable on the same scale."""
    conds = [c for c in cond_order if c in bar_agg_group]
    n = len(conds)
    x = np.arange(n)
    means = [bar_agg_group[c].get("mean", np.nan) for c in conds]
    cis   = [bar_agg_group[c].get("ci95", 0.0) for c in conds]

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.bar(x, means, 0.6, yerr=cis,
           error_kw={"elinewidth": err_lw, "capthick": err_capthick, "capsize": err_capsize},
           color=color, label=label)

    ax.set_yscale("log")
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v)}" if v >= 1 else f"{v:.1f}"))
    ax.set_xticks(x)
    ax.set_xticklabels([cond_labels[c] for c in conds], fontsize=13)
    ax.set_ylabel(r"Selection latency (ms/frame)", fontsize=18)
    ax.tick_params(labelsize=16)

    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)
    ax.legend(fontsize=14, framealpha=0.9, loc="upper left")

    fig.set_constrained_layout(True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)


def plot_timing_stages(stage_means_by_cond, cond_order, cond_labels, out_path):
    """Per-stage stacked bar with a broken y-axis (oracle ~2500ms dwarfs the rest, <300ms).
    Left/right spines stay closed on both panels (standard broken-axis convention) --
    only the shared top/bottom seam is cut, marked by the diagonal break ticks. No
    figure title (synth vs. real is in the filename/caption, not baked into the PNG)."""
    conds = [c for c in cond_order if c in stage_means_by_cond]
    totals = [sum(stage_means_by_cond[c].values()) for c in conds]
    non_oracle_max = max(t for c, t in zip(conds, totals) if c != "tiled_oracle")
    break_y = non_oracle_max * 1.35
    top_max = max(totals) * 1.22  # extra headroom so the top value label doesn't clip the spine

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7.5, 6.0),
        gridspec_kw={"height_ratios": [1, 2.2], "hspace": 0.08},
        constrained_layout=False,
    )

    n = len(conds)
    x = np.arange(n)
    w = 0.55
    for ax in (ax_top, ax_bot):
        bottoms = np.zeros(n)
        for sk, label, color in TIMING_STAGE_DEFS:
            heights = np.array([stage_means_by_cond[c][sk] for c in conds])
            ax.bar(x, heights, w, bottom=bottoms, color=color, label=label)
            bottoms += heights

    ax_bot.set_ylim(0, break_y)
    ax_top.set_ylim(break_y, top_max)

    for i, total in enumerate(totals):
        if total > break_y:
            ax, off = ax_top, (top_max - break_y) * 0.03
        else:
            ax, off = ax_bot, break_y * 0.02
        ax.text(i, total + off, f"{total:.1f}", ha="center", va="bottom", fontsize=11, color="#333333")

    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels([cond_labels[c] for c in conds], fontsize=12)
    ax_bot.set_ylabel(r"Selection latency (ms/frame)", fontsize=14)
    ax_bot.tick_params(axis="y", labelsize=13)
    ax_top.tick_params(axis="y", labelsize=13)
    ax_bot.tick_params(axis="x", length=0)

    ax_top.spines["bottom"].set_visible(False)
    ax_bot.spines["top"].set_visible(False)
    ax_top.tick_params(axis="x", bottom=False, labelbottom=False)

    d = 0.012
    kw = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_top.plot((-d, +d), (-2 * d, +2 * d), **kw)
    kw = dict(transform=ax_bot.transAxes, color="k", clip_on=False, linewidth=1.2)
    ax_bot.plot((-d, +d), (1 - d, 1 + d), **kw)

    # legend goes in ax_top's blank region (only the rightmost/oracle bar has content there)
    handles = [mpatches.Patch(color=c, label=lbl) for _, lbl, c in TIMING_STAGE_DEFS]
    ax_top.legend(handles=handles, loc="upper left", ncol=2, fontsize=10, framealpha=0.9)

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_path.with_suffix(".eps"), format="eps", bbox_inches="tight")
    print(f"Wrote {out_path}")
    plt.close(fig)

In [22]:
# ylim computed per-sweep (not shared across grid8/grid2/grid4 -- grid2/4's dynamic
# range is much smaller than grid8's, sharing grid8's scale would flatten the bars).
for _suffix, _bar_agg in timing_bar_aggs.items():
    _out_dir = TIMING_SWEEPS[_suffix][1]
    _all_lo = [v["mean"] - v["ci95"] for g in _bar_agg.values() for v in g.values()]
    _all_hi = [v["mean"] + v["ci95"] for g in _bar_agg.values() for v in g.values()]
    _ylim = (min(_all_lo) * 0.7, max(_all_hi) * 1.4)

    for group, title in TIMING_GROUP_LABELS.items():
        plot_timing_bar(_bar_agg[group], TIMING_COND_ORDER, TIMING_COND_LABELS,
                         color_palette[0], title, _out_dir / f"selection_timing_bar_{group}{_suffix}.png",
                         ylim=_ylim)

        stage_means = timing_stage_group_means(group, df=timing_dfs[_suffix])
        plot_timing_stages(stage_means, TIMING_COND_ORDER, TIMING_COND_LABELS,
                            _out_dir / f"selection_timing_stages_{group}{_suffix}.png")

print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_bar_synth.png


/tmp/ipykernel_1979135/1798321835.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/timings_selection/selection_timing_stages_synth.png


...

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0703_timing_grid_retry/selection_timing_bar_real_grid4.png


/tmp/ipykernel_1979135/1798321835.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote output/quickplots/0703_timing_grid_retry/selection_timing_stages_real_grid4.png
Done.


## Exp 4 — Granularity (tile-count sweep)

Source: `output/0704/quality_sweep/summary_all_ml_fixed.csv` (merged: old non-ml rows
from the 2026-07-09 consolidated sweep + fixed-label `ml` rows from
`output/0709/quality_sweep_ml_retrain/` after the `ml/features.py` label-censoring fix
+ pooled-model retrain -- see PLAN.md Open item 11). grid in {1,2,4,8,16}³, packing in
{progressive, tile_partial, tile_strict}. Progressive only has `vd_lod` (pure GS
ordering, no tiling); tiled packings run all 4 schemes. Same two-stage scene-then-CI
aggregation as Exp1/Exp2 (`_aggregate`), with `grid` as an extra group-by dimension.
PSNR/SSIM/LPIPS all plotted (LPIPS enabled for this sweep via `--lpips`; not yet
available for Exp1/Exp2, which predate the all-experiments-compute-LPIPS rule).

This grid sweep also answers Exp3 (3A/3B), no separate rerun needed:
- **3A (tile_partial vs tile_strict):** `delta_partial_vs_strict_{psnr,ssim,lpips}.png`
  below -- partial-minus-strict gap per grid size.
- **3B (culling benefit):** `psnr_vs_budget_progressive.png` / `..._ssim_...` /
  `..._lpips_...` below -- progressive packing's tile-visibility gate only has teeth once
  tiles are smaller than the scene; grid1³ is a single scene-spanning tile (the gate never
  fires, i.e. the "no-cull" limit), grid16³ is the most-culled point on the same line.


In [23]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# 2026-07-09: ml/features.py had a label-censoring bug (mse_loo<=0 rows silently
# NaN-dropped instead of floored) that starved the ml scheme's training data on real
# scenes at fine grids. Fixed + all pooled models retrained + ml scheme rerun
# (output/0709/quality_sweep_ml_retrain/). This merged CSV = old non-ml rows (unaffected
# by the bug) + the new fixed ml rows -- the comprehensive, correct Exp4 dataset.
EXP4_SUMMARY_CSV = "output/0704/quality_sweep/summary_all_ml_fixed.csv"
EXP4_OUT_DIR     = Path("plotting/paper/exp4_granularity/")
EXP4_BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]
EXP4_SCHEME_ORDER = ["vd_lod", "v_lod_w", "ml", "oracle_loo"]

GRID_ORDER  = [1, 2, 4, 8, 16]
GRID_CONFIG = {
    1:  {"label": r"$1^3$",  "marker": "o", "color": "#d62728"},
    2:  {"label": r"$2^3$",  "marker": "s", "color": "#ff7f0e"},
    4:  {"label": r"$4^3$",  "marker": "^", "color": "#2ca02c"},
    8:  {"label": r"$8^3$",  "marker": "D", "color": "#1f77b4"},
    16: {"label": r"$16^3$", "marker": "P", "color": "#9467bd"},
}

exp4_csv = Path(EXP4_SUMMARY_CSV)
if not exp4_csv.exists():
    raise FileNotFoundError(f"{exp4_csv}  (cwd={Path.cwd()})")

exp4_df = pd.read_csv(exp4_csv)
# exp1's weight-mode slice reuses grid8/progressive/vd_lod (volume/volume_over_d2/random
# on top of screen_area) -- keep only screen_area or grid8/progressive is 4x-inflated.
exp4_df = exp4_df[exp4_df["weight_mode"] == "screen_area"].copy()
exp4_df["grid"] = exp4_df["grid_shape"].str.extract(r"\[(\d+)").astype(int)

# provenance: exact sliced rows behind this experiment's figures, alongside the PNGs
EXP4_OUT_DIR.mkdir(parents=True, exist_ok=True)
exp4_df.to_csv(EXP4_OUT_DIR / "source_summary.csv", index=False)

exp4_rows = exp4_df.to_dict("records")
exp4_rows = _apply_budget_labels(exp4_rows, EXP4_BUDGET_PCTS)

print(f"Loaded {len(exp4_rows)} rows | packing_mode: {sorted(exp4_df['packing_mode'].unique())} "
      f"| grid: {sorted(exp4_df['grid'].unique())} | scheme: {sorted(exp4_df['scheme'].unique())}")


Loaded 540000 rows | packing_mode: ['progressive', 'tile_partial', 'tile_strict'] | grid: [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)] | scheme: ['ml', 'oracle_loo', 'v_lod_w', 'vd_lod']


In [24]:
def _exp4_agg_by_grid(rows, packing_mode, scheme=None):
    """Filter to one packing_mode (+ optional scheme), aggregate grouped by grid."""
    filtered = [r for r in rows if r["packing_mode"] == packing_mode
                and (scheme is None or r["scheme"] == scheme)]
    return _aggregate(filtered, "grid")


def _exp4_agg_by_scheme(rows, packing_mode, grid):
    """Filter to one (packing_mode, grid), aggregate grouped by scheme."""
    filtered = [r for r in rows if r["packing_mode"] == packing_mode and r["grid"] == grid]
    return _aggregate(filtered, "scheme")


EXP4_OUT_DIR.mkdir(parents=True, exist_ok=True)
(EXP4_OUT_DIR / "_grid").mkdir(parents=True, exist_ok=True)

# 1. Progressive (pure GS ordering, no tiling) -- single panel, lines = grid size
prog_agg = _exp4_agg_by_grid(exp4_rows, "progressive")
plot_metric(prog_agg, GRID_ORDER, "psnr", r"Quality in PSNR (dB)",
            "psnr_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(prog_agg, GRID_ORDER, "ssim", r"Quality in SSIM",
            "ssim_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)
plot_metric(prog_agg, GRID_ORDER, "lpips", r"LPIPS (lower is better)",
            "lpips_vs_budget_progressive", EXP4_OUT_DIR, key_config=GRID_CONFIG)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/psnr_vs_budget_progressive.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/ssim_vs_budget_progressive.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/lpips_vs_budget_progressive.{png,eps}


In [25]:
# 2. Tiled packings -- one panel per scheme, lines = grid size
for packing in ["tile_partial", "tile_strict"]:
    panels_agg = {
        KEY_CONFIG[s]["label"]: _exp4_agg_by_grid(exp4_rows, packing, scheme=s)
        for s in EXP4_SCHEME_ORDER
    }
    plot_grid(panels_agg, GRID_ORDER, "psnr", r"Quality in PSNR (dB)",
              EXP4_OUT_DIR / "_grid" / f"psnr_vs_budget_{packing}.png",
              ncols=4, key_config=GRID_CONFIG)
    plot_grid(panels_agg, GRID_ORDER, "ssim", r"Quality in SSIM",
              EXP4_OUT_DIR / "_grid" / f"ssim_vs_budget_{packing}.png",
              ncols=4, key_config=GRID_CONFIG)
    plot_grid(panels_agg, GRID_ORDER, "lpips", r"LPIPS (lower is better)",
              EXP4_OUT_DIR / "_grid" / f"lpips_vs_budget_{packing}.png",
              ncols=4, key_config=GRID_CONFIG)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/psnr_vs_budget_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_vs_budget_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp4_granularity/_grid/psnr_vs_budget_tile_strict.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_vs_budget_tile_strict.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/lpips_vs_budget_tile_strict.png


In [26]:
# 3. 8^3 vs 16^3 head-to-head -- one panel per grid, lines = scheme
for packing in ["tile_partial", "tile_strict"]:
    panels_agg = {
        GRID_CONFIG[g]["label"]: _exp4_agg_by_scheme(exp4_rows, packing, g)
        for g in (8, 16)
    }
    plot_grid(panels_agg, EXP4_SCHEME_ORDER, "psnr", r"Quality in PSNR (dB)",
              EXP4_OUT_DIR / "_grid" / f"psnr_8v16_{packing}.png", ncols=2)
    plot_grid(panels_agg, EXP4_SCHEME_ORDER, "ssim", r"Quality in SSIM",
              EXP4_OUT_DIR / "_grid" / f"ssim_8v16_{packing}.png", ncols=2)
    plot_grid(panels_agg, EXP4_SCHEME_ORDER, "lpips", r"LPIPS (lower is better)",
              EXP4_OUT_DIR / "_grid" / f"lpips_8v16_{packing}.png", ncols=2)


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/psnr_8v16_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_8v16_tile_partial.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/exp4_granularity/_grid/psnr_8v16_tile_strict.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/ssim_8v16_tile_strict.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/_grid/lpips_8v16_tile_strict.png


In [27]:
# 4. tile_partial - tile_strict gap, pooled across schemes, lines = grid size
partial_pooled = _aggregate([r for r in exp4_rows if r["packing_mode"] == "tile_partial"], "grid")
strict_pooled  = _aggregate([r for r in exp4_rows if r["packing_mode"] == "tile_strict"],  "grid")


def _plot_delta(metric, ylabel, out_stem):
    fig, ax = plt.subplots(figsize=figsize)
    for g in GRID_ORDER:
        if g not in partial_pooled or g not in strict_pooled:
            continue
        bks = sorted(set(partial_pooled[g]) & set(strict_pooled[g]), key=_bk_sort)
        xs  = [_bk_sort(b) for b in bks]
        ys  = [partial_pooled[g][b][f"{metric}_mean"] - strict_pooled[g][b][f"{metric}_mean"] for b in bks]
        cfg = GRID_CONFIG[g]
        ax.plot(xs, ys, marker=cfg["marker"], color=cfg["color"], linewidth=2.5,
                markersize=8, label=cfg["label"])
    ax.axhline(0.0, color="gray", linestyle=":", linewidth=1.0, zorder=0)
    ax.set_xlabel(r"Budget (\% of scene)", fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x)}\\%"))
    ax.legend(loc="best", framealpha=0.9, fontsize=16, title="grid")
    ax.tick_params(labelsize=18)
    fig.set_constrained_layout(True)
    fig.savefig(EXP4_OUT_DIR / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(EXP4_OUT_DIR / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {EXP4_OUT_DIR}/{out_stem}.{{png,eps}}")
    plt.close(fig)


_plot_delta("psnr", r"$\Delta$ PSNR: tile\_partial $-$ tile\_strict (dB)", "delta_partial_vs_strict_psnr")
_plot_delta("ssim", r"$\Delta$ SSIM: tile\_partial $-$ tile\_strict", "delta_partial_vs_strict_ssim")
_plot_delta("lpips", r"$\Delta$ LPIPS: tile\_partial $-$ tile\_strict", "delta_partial_vs_strict_lpips")
print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/delta_partial_vs_strict_psnr.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/delta_partial_vs_strict_ssim.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/exp4_granularity/delta_partial_vs_strict_lpips.{png,eps}
Done.


## ML scheme comparison — pooled-all10 vs domain-split pooled vs LGBM-rank (2026-07-13)

Grid8/tile_partial/screen_area only. 6 `model_arm`s: `all10_reg` (deployed
`pooled_all10_ACG`, reused from `output/0709/quality_sweep_ml_retrain` -- no code changed
in the regressor path since 07-09, rerunning would reproduce identical numbers) plus 5
freshly-run arms (`real3_reg`, `synth7_reg`, `all10_rank`, `real3_rank`, `synth7_rank` --
see `.claude/runs.md` history / `experiments/0713/rank_vs_pool_comparison_sweep.sh`). Rank
arms use `LGBMRanker` with direct rank-order sort (`--greedy-key utility` auto-override,
no byte-divisible calibration). Two panel sets, split by domain: real3 scenes
(bicycle/garden/stump) and synth7 scenes -- each domain only shows the arms trained/evaluable
on it (`all10_*` appears in both, `real3_*`/`synth7_*` only in their own domain).

In [28]:
# ── I/O ──────────────────────────────────────────────────────────────────────
OUT_DIR     = "plotting/paper/rank_vs_pool_comparison/"
GROUP_BY    = "model_arm"
BUDGET_PCTS = [10, 25, 40, 55, 70, 85, 99, 100]

REAL3_SCENES  = ["bicycle", "garden", "stump"]
SYNTH7_SCENES = ["chair", "drums", "ficus", "hotdog", "materials", "mic", "ship"]

ARM_CSVS = {
    "all10_reg":  "output/0709/quality_sweep_ml_retrain/summary_all.csv",
    "real3_reg":  "output/0713/rank_vs_pool_comparison/real3_reg/summary_all.csv",
    "synth7_reg": "output/0713/rank_vs_pool_comparison/synth7_reg/summary_all.csv",
    "all10_rank": "output/0713/rank_vs_pool_comparison/all10_rank/summary_all.csv",
    "real3_rank": "output/0713/rank_vs_pool_comparison/real3_rank/summary_all.csv",
    "synth7_rank": "output/0713/rank_vs_pool_comparison/synth7_rank/summary_all.csv",
}

parts = []
for arm, csv_path in ARM_CSVS.items():
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"{p}  (cwd={Path.cwd()})")
    sub = pd.read_csv(p)
    sub = sub[(sub["grid_shape"] == "[8, 8, 8]") & (sub["packing_mode"] == "tile_partial")
              & (sub["weight_mode"] == "screen_area") & (sub["scheme"] == "ml")].copy()
    sub["model_arm"] = arm
    parts.append(sub)
df_all = pd.concat(parts, ignore_index=True)

# provenance: exact sliced rows behind this comparison's figures, alongside the PNGs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
df_all.to_csv(Path(OUT_DIR) / "source_summary.csv", index=False)

df_real3  = df_all[df_all["scene"].isin(REAL3_SCENES)].copy()
df_synth7 = df_all[df_all["scene"].isin(SYNTH7_SCENES)].copy()

rows_real3  = _apply_budget_labels(df_real3.to_dict("records"), BUDGET_PCTS)
rows_synth7 = _apply_budget_labels(df_synth7.to_dict("records"), BUDGET_PCTS)

print(f"real3:  {len(rows_real3)} rows | arms: {sorted(set(r['model_arm'] for r in rows_real3))}")
print(f"synth7: {len(rows_synth7)} rows | arms: {sorted(set(r['model_arm'] for r in rows_synth7))}")


real3:  14400 rows | arms: ['all10_rank', 'all10_reg', 'real3_rank', 'real3_reg']
synth7: 33600 rows | arms: ['all10_rank', 'all10_reg', 'synth7_rank', 'synth7_reg']


In [29]:
ARM_KEY_CONFIG = {
    "all10_reg":  {"label": "Regressor (pooled-all10, deployed)", "marker": "s", "color": color_palette[0]},
    "real3_reg":  {"label": "Regressor (real3-pooled)",           "marker": "^", "color": color_palette[2]},
    "synth7_reg": {"label": "Regressor (synth7-pooled)",          "marker": "^", "color": color_palette[2]},
    "all10_rank": {"label": "Ranker (pooled-all10)",              "marker": "P", "color": color_palette[3]},
    "real3_rank": {"label": "Ranker (real3-pooled)",              "marker": "*", "color": color_palette[1]},
    "synth7_rank": {"label": "Ranker (synth7-pooled)",            "marker": "*", "color": color_palette[1]},
}
ARM_ORDER_REAL3  = ["all10_reg", "real3_reg", "all10_rank", "real3_rank"]
ARM_ORDER_SYNTH7 = ["all10_reg", "synth7_reg", "all10_rank", "synth7_rank"]


In [30]:
agg_real3  = _aggregate(rows_real3, GROUP_BY)
agg_synth7 = _aggregate(rows_synth7, GROUP_BY)

order_real3  = [k for k in ARM_ORDER_REAL3 if k in agg_real3]
order_synth7 = [k for k in ARM_ORDER_SYNTH7 if k in agg_synth7]

from collections import defaultdict as _dd
_by_scene_real3  = _dd(list)
_by_scene_synth7 = _dd(list)
for r in rows_real3:
    _by_scene_real3[r["scene"]].append(r)
for r in rows_synth7:
    _by_scene_synth7[r["scene"]].append(r)
scenes_agg_real3  = {s: _aggregate(_by_scene_real3[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene_real3}
scenes_agg_synth7 = {s: _aggregate(_by_scene_synth7[s], GROUP_BY) for s in SCENE_ORDER if s in _by_scene_synth7}

print(f"real3 order:  {order_real3}")
print(f"synth7 order: {order_synth7}")


real3 order:  ['all10_reg', 'real3_reg', 'all10_rank', 'real3_rank']
synth7 order: ['all10_reg', 'synth7_reg', 'all10_rank', 'synth7_rank']


In [31]:
out_dir  = Path(OUT_DIR)
grid_dir = out_dir / "_grid"

# real3 domain
for metric, ylabel in [("psnr", r"Quality in PSNR (dB)"), ("ssim", r"Quality in SSIM"), ("lpips", r"LPIPS (lower is better)")]:
    plot_metric(agg_real3, order_real3, metric, ylabel, f"real3_{metric}_vs_budget", out_dir, key_config=ARM_KEY_CONFIG)
    plot_grid(scenes_agg_real3, order_real3, metric, ylabel, grid_dir / f"real3_{metric}_vs_budget.png", ncols=3, key_config=ARM_KEY_CONFIG)

# synth7 domain
for metric, ylabel in [("psnr", r"Quality in PSNR (dB)"), ("ssim", r"Quality in SSIM"), ("lpips", r"LPIPS (lower is better)")]:
    plot_metric(agg_synth7, order_synth7, metric, ylabel, f"synth7_{metric}_vs_budget", out_dir, key_config=ARM_KEY_CONFIG)
    plot_grid(scenes_agg_synth7, order_synth7, metric, ylabel, grid_dir / f"synth7_{metric}_vs_budget.png", ncols=4, key_config=ARM_KEY_CONFIG)

print("Done.")


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/rank_vs_pool_comparison/real3_psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/rank_vs_pool_comparison/_grid/real3_psnr_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Wrote plotting/paper/rank_vs_pool_comparison/_grid/synth7_ssim_vs_budget.png


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/rank_vs_pool_comparison/synth7_lpips_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote plotting/paper/rank_vs_pool_comparison/_grid/synth7_lpips_vs_budget.png
Done.
